# Connectivity Analysis

Jupyter notebook containing pipeline for calling connectivity on single cells infected with RVdG

Input for this notebook requires:
1) Transcriptomic assignments recorded in mapped_centroids.csv (derived from 09_centroid_mapping.py)
2) RVdG count matrices derived from 11_experiment_barcode_alignment.sh
3) Helper count matrices derived from 11_experiment_barcode_alignment.sh
4) Viral barcode libraries derived from 01_barcode_diversity_alignment.sh

Output includes:
1) processed_barcodes_df, which includes all RVdG barcodes+cells passing QC
2) metadata_df, a thresholded dataframe of cell type annotations
3) Plots related to QC metrics, including SBARRO thresholds and distribution of starter/presynaptic cells across networks
4) Upset plots showing experimental barcode diversity across slices
5) Plots related to identifying cell type-specific connectivity, including graph networks, enrichment matrices, motif heatmaps, and a dataframe for generating a Sankey

Modules and their versions used when generating figures for the paper can be found in 'requirements.txt', which is stored in our GitHub repository: https://github.com/MEUrbanek/rabies_barcode_tech

This code was last amended by Maddie Urbanek on 2026/02/19

Code reviewer: Ike Ogbonna, 2026/02/23

## Notebook set-up and function definitions

In [ ]:
# For module versions, see requirements.txt on GitHub linked above
import ast
import os
import time

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import networkx as nx
from scipy import stats
from scipy.stats import chi2
import seaborn as sns
from upsetplot import from_contents, plot


# Change path name in function below to top-most directory containing data
os.chdir(
    # '/Users/maddieurbanek/Desktop/revision_data/resubmission/data/')
    '/Users/ikogbonna/Documents/Code/code review/')
    # '/Users/ike/Documents/Code/code review/')

for d in ('./figs/sfig_conn', './connectivity/matrices'):
    os.makedirs(d, exist_ok=True)

#Font formatting for exporting plots
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')

dec_list = ['s1','s2','s3','s4','s5']
mar_list = ['c1','c2','n1','n2','c3','c4','n3','n4']

# Define broad classes for downstream annotations
en = [
    'EN-L5_6-NP', 'EN-L4-IT','EN-Newborn', 'EN-L2_3-IT','EN-L6b','EN-IT-Immature','EN-L5-ET', 'EN-L6-CT', 'EN-L6-IT','EN-Non-IT-Immature','EN-L5-IT','Cajal-Retzius cell']
inh = [
    'IN-NCx_dGE-Immature','IN-CGE-Immature','IN-MGE-SST','IN-CGE-VIP', 'IN-CGE-SNCG', 'IN-MGE-PV','IN-Mix-LAMP5','IN-MGE-Immature']
pro = ['IPC-EN','Tri-IPC']
glia = [
    'Astrocyte-Fibrous','Microglia','OPC','Vascular','Astrocyte-Protoplasmic','RG-vRG', 'Oligodendrocyte','Oligodendrocyte-Immature','Astrocyte-Immature','RG-tRG','RG-oRG']
unknown = ['Unknown']

# Define subclasses (excitatory layer identites, glial types, inhibitory regions)
astro = ['Astrocyte-Fibrous','Astrocyte-Protoplasmic','Astrocyte-Immature']
opc = ['OPC','Oligodendrocyte','Oligodendrocyte-Immature']
rgc = ['RG-tRG','RG-oRG','RG-vRG']

mge = ['IN-MGE-SST','IN-MGE-PV','IN-MGE-Immature']
cge = ['IN-CGE-Immature','IN-CGE-VIP', 'IN-CGE-SNCG']
dge = ['IN-NCx_dGE-Immature']

immature = ['EN-Newborn','EN-IT-Immature','EN-Non-IT-Immature']
deep = ['EN-L5_6-NP','EN-L6b','EN-L5-ET','EN-L6-CT', 'EN-L6-IT','EN-L5-IT']

pivot_types = [
    'RG','IPC','EN-Immature','EN-L2_3-IT','EN-L4-IT','EN-Deep Layer','Cajal-Retzius cell','IN-CGE','IN-MGE','IN-DGE']

In [ ]:
def umi_threshold(
        thresholded_data,
        umi: int = 1
):
    """
    Parameters
    ----------
    thresholded_data
        split barcode matrices
    umi
        minimum number of UMIs needed per cell to pass threshold, default keep all barcodes
    """
    pre_count = thresholded_data['barcode'].nunique()
    thresholded_data = thresholded_data.loc[thresholded_data['UMI_Count'] >= umi]
    post_count = thresholded_data['barcode'].nunique()
    
    print(
        f'Total number of unique barcodes before thresholding\n{pre_count}',
        f'\nTotal number of unique barcodes after thresholding\n{post_count}\n')
    return thresholded_data


def calculate_fi_trust(
        sample_dataset,
        viral_library,
        bootstraps: int = 10
):
    """Calculate number of draws from viral barcode library needed to match number of unique barcodes observed in at least 2 cells per experiment
    
    Parameters
    ----------
    sample_dataset
        barcode count matrix of transcriptome-passing, UMI thresholded cells
    viral_library
        diversity library that matches the rabies used to infect the slice, row per barcode UMI
    bootstraps
        number of iterations for infection simulations
    """
    start_time = time.time()
    print('Calculating Founder Infection Trust Threshold')
    
    #Get total number of unique barcodes observed in the slice
    dup_counts = sample_dataset['barcode'].value_counts().reset_index()
    dup_counts = dup_counts.loc[dup_counts['count'] >= 2]
    total_obs_bc = dup_counts['barcode'].nunique()
    print(
        f'N unique barcodes detected in ≥ 2 cells per slice:\n{total_obs_bc}',
        'Beginning VBC draws')

    # Bootstrap FI trust score by simulating barcode draws until the same amount of unique barcodes as observed n the original dataset are found
    fi_trust = []
    for i in range(bootstraps):
        pulled_vbc = pd.DataFrame(columns=['original_barcode'])
        while True:
            acquired_vbc = pd.DataFrame(
                np.random.choice(viral_library['original_barcode'], 1, replace=True), columns=['original_barcode'])
            pulled_vbc = pd.concat([pulled_vbc,acquired_vbc], ignore_index=True, axis=0)
            if pulled_vbc['original_barcode'].nunique() >= total_obs_bc:
                fi_trust.append(len(pulled_vbc))
                break
    
    fi_trust_score = np.median(fi_trust)
    print(
        f'Done calculating FI Trust Threshold! :)\n{fi_trust}',
        f'\nFI Trust Threshold = {fi_trust_score}',
        f'\nThis took... {time.time() - start_time} seconds!')
    return fi_trust_score


def threshold_fi_trust(
        experimental_library,
        experimental_trust_score,
        viral_library,
        p: float = 0.9
):
    """Make function for thresholding fi trust score by dataset
    
    Parameters
    ----------
    experimental_library
        individual slice/experiment being tested
    experimental_trust_score
        score calculated from the calculate_fi_trust() function
    viral_library
        unexploded viral complexity library with barcode sequences and proportions
    p : float
        acceptable risk of a barcode entering the system 2x. Default from Shin & Urbanek et al.
    """
    exp_count = experimental_library['barcode'].nunique()
    print(
        'Thresholding experimental dataset based on FI trust scores',
        f'\nN unique barcodes from experimental dataset: {exp_count}')

    # Make list of barcodes that aren't meeting the threshold
    threshold_viral = viral_library[['original_barcode', f'p = {p}']]
    threshold_viral = threshold_viral.loc[threshold_viral[f'p = {p}'] <= experimental_trust_score]
    thresholded_library = experimental_library.loc[
        ~(experimental_library['barcode'].str[3:].isin(threshold_viral['original_barcode']))]

    thresh_count = thresholded_library['barcode'].nunique()
    print(
        f'N unique barcodes retained after thresholding on FI trust scores: {thresh_count}',
        f'\nPercentage of barcodes retained: {thresh_count / exp_count * 100}%')
    return thresholded_library


def process_barcodes_df(
        metadata_df,
        feature_column,
        barcodes_df,
        helpers_df
):
    """
    Parameters
    ----------
    metadata_df
        filtered metadata dataframe generated above
    feature_column
        column name in metadata_df that you want to use for marking annotations
    barcodes_df
        filtered RVdG barcode dataframe generated above
    helpers_df
        thresholded helper dataframe generated above
    """
    # Add celltype metadata
    barcodes_df['celltype'] = barcodes_df['CBC'].map(metadata_df.set_index('cellbarcode')[feature_column])

    # Identify barcodes in ≥2 cells
    barcodes_df['duplicate?'] = barcodes_df.duplicated(subset=['barcode'], keep=False)

    # Assign starters
    barcodes_df['helper'] = 'nonstarter'
    barcodes_df.loc[barcodes_df['CBC'].isin(helpers_df['CBC']), 'helper'] = 'starter'
    barcodes_df['helper_UMI_count'] = barcodes_df['CBC'].map(dict(zip(helpers_df['CBC'], helpers_df['UMI_Count']))).fillna(0)

    # Label cells that are in single-starter networks
    starter_barcodes_df = barcodes_df.loc[barcodes_df['helper'] == 'starter']
    barcodes_df['starter_barcode'] = 'n'
    barcodes_df.loc[(barcodes_df['barcode'].isin(starter_barcodes_df['barcode'])), 'starter_barcode'] = 'y'
    starter_barcodes_df_counts = starter_barcodes_df.groupby('barcode').count()['CBC'].reset_index()
    single_starter_barcodes = starter_barcodes_df_counts.loc[starter_barcodes_df_counts['CBC'] == 1, 'barcode']
    barcodes_df['single_starter_barcode'] = 'n'
    barcodes_df.loc[barcodes_df['barcode'].isin(single_starter_barcodes), 'single_starter_barcode'] = 'y'

    # determining and labeling nonstarter barcodes:
    barcodes_df['non_starter_barcode'] = 'n'
    nonstarter_barcodes_list = barcodes_df.loc[barcodes_df['helper'] == 'nonstarter', 'barcode']
    barcodes_df.loc[barcodes_df['barcode'].isin(nonstarter_barcodes_list), 'non_starter_barcode'] = 'y'
    barcodes_df['starter/nonstarter barcode'] = (
        barcodes_df['non_starter_barcode'] + barcodes_df['starter_barcode']).map({'yy': 'both', 'yn': 'nonstarter_only', 'ny': 'starter_only', 'nn': 'error'})
    return barcodes_df


def build_null_matrix(
        non_starter_proportions,
        experimental_dataset,
        metadata_df,
        annotations: str = 'celltype',
        plot_heatmaps: bool = False
):
    """
    Parameters
    ----------
    non_starter_proportions
        non-starter proportions generated from uninfected datasets
    experimental_dataset
        dataset-split (or pooled) w/ starter cell_type and starter CBC columns
    metadata_df
    annotations : str, optional
        which column of annotations to use. One of 'broad_class', 'subclass', 'celltype'
    plot_heatmaps : bool, optional
        decides whether to plot heatmaps for nulls
    """
    # Drop identical CBCs and renamce columns
    starters = experimental_dataset.drop_duplicates(
        subset='starter CBC').rename(columns={'starter cell_type': 'starter_cell_type'})
    starters['starter_cell_type'] = starters['starter CBC'].map(
        metadata_df.set_index('cellbarcode')[annotations])

    starter_proportions = starters.groupby('starter_cell_type').count()[
        'datasetid'].reset_index().rename(columns={'datasetid': 'number of cells'})
    starter_proportions['proportion of celltype in starter cells'] = (
        starter_proportions['number of cells'] / starter_proportions['number of cells'].sum())

    # Build null
    null_df = non_starter_proportions.merge(starter_proportions, how='cross')

    # Calculate proportion of connections for each non-starter to starter combo
    null_df['proportion of connections'] = (
        null_df['proportion of celltype in nonstarter cells'] * null_df['proportion of celltype in starter cells'])

    # Create column of presynaptic to postsynaptic cell type connections
    null_df['pre-post'] = null_df['non_starter_cell_type'] + "+" + null_df[
        'starter_cell_type']

    if plot_heatmaps:
        pivot = null_df.pivot(
            index='non_starter_cell_type', columns='starter_cell_type')['proportion of connections']
        # pivot = pivot.reindex(
        #     ["RG","OPCs","Astrocytes","Progenitors","Interneurons","L2/3","L4","L5/6/SP"], level=0).T.reindex(["RG","OPCs","Astrocytes","Progenitors","Interneurons","L2/3","L4","L5/6/SP"]).T

        # Plot null matrix heatmap (without log enrichments, this is just proportions)
        plt.figure(figsize=(15, 8))
        sns.heatmap(pivot, annot=False, cmap='Reds', square=True, vmin=0, vmax=1)
        plt.ylabel('Non-starter')
        plt.xlabel('Starter')
        # plt.savefig('./null_matrices/null.svg', bbox_inches='tight', format='svg')

    return null_df


def build_obs_matrix(
        conn_1_compiling,
        null_df,
        metadata_df,
        annotations: str = 'celltype',
        plot_heatmaps: bool = False
):
    """
    Parameters
    ----------
    conn_1_compiling
        conn_1_compiling dataframes
    null_df
        dataset-specific null dataframes
    metadata_df
    annotations : str, optional
        specify what level of type annotations to use. One of 'broad_class', 'subclass', 'celltype'
    plot_heatmaps : bool, optional
        decides whether to plot heatmaps for nulls
    """
    feature_dict = metadata_df.set_index('cellbarcode')[annotations]
    conn_1_compiling['starter cell_type'] = conn_1_compiling['starter CBC'].map(
        feature_dict)
    conn_1_compiling['non-starter cell_type'] = conn_1_compiling['non-starter CBC'].map(
        feature_dict)

    # Calculate the total number of each non-starter to starter cell type combination
    conn_matrix_df = conn_1_compiling.groupby(
        ['non-starter cell_type', 'starter cell_type']).count()['conn_type'].reset_index().rename(columns={'conn_type': 'observed connections'})

    # Make column with presynaptic to postsynaptic cell type combos
    conn_matrix_df['pre-post'] = (
        conn_matrix_df['non-starter cell_type'] + '+' + conn_matrix_df['starter cell_type'])

    # Make empty entries for unobserved cell type combinations in the null matrix
    missing_connections = list(
        (set(null_df['pre-post'])).difference(set(conn_matrix_df['pre-post'])))
    df_new = pd.DataFrame({
        'pre-post': missing_connections,
        'observed connections': [0] * len(missing_connections),
        'non-starter cell_type': [x.split('+')[0] for x in missing_connections],
        'starter cell_type': [x.split('+')[1] for x in missing_connections]})

    # Concatenate both matrices so you have a matrix that's the same size as the null 
    conn_matrix_df = pd.concat([conn_matrix_df, df_new])

    # Calculate the proportion of each pre-post cell type connection
    conn_matrix_df['proportion of connections'] = (
        conn_matrix_df['observed connections'] / conn_matrix_df['observed connections'].sum())

    if plot_heatmaps:
        # Plot observed matrix heatmap (without log enrichments, this is just proportions)
        plt.figure(figsize=(15, 8))
        pivot = conn_matrix_df.pivot(
            index='non-starter cell_type', columns='starter cell_type')['proportion of connections']
        # pivot = pivot.reindex(
        #     ["RG","OPCs","Astrocytes","Progenitors","Interneurons","L2/3","L4","L5/6/SP"], level=0).T.reindex(["RG","OPCs","Astrocytes","Progenitors","Interneurons","L2/3","L4","L5/6/SP"]).T
        sns.heatmap(pivot, annot=False, cmap='Reds', square=True, vmin=0, vmax=1)
        plt.ylabel('Non-starter')
        plt.xlabel('Starter')
        # plt.savefig('./observed_matrices/obs.svg', bbox_inches='tight', format='svg')

    return conn_matrix_df


def check_minimum_expected_connections(
        observed,
        null,
        acceptable_min
):
    """
    Parameters
    ----------
    observed
        observed matrix from build_obs_matrix()
    null
        null matrix from build_null_matrix()
    acceptable_min
        minimum number of expected connections to run chi-squared on
    """
    # Add any missing connections that aren't in the uninfected datasets 
    df_new = pd.DataFrame({
        'pre-post': list((set(observed['pre-post'])).difference(set(null['pre-post']))),
        'proportion of connections': 0})

    # Concatenate both matrices so you have a matrix that's the same size as the observed 
    null = pd.concat([null, df_new])
    
    # Links the proportions from null matrix to matching cell type combos in the observed
    observed['null proportion of connections'] = observed['pre-post'].map(
        null.set_index('pre-post')['proportion of connections'])

    #Calculate scaled expected connections
    observed['expected connections'] = (
        observed['observed connections'].sum() * observed['null proportion of connections'])
    print(
        'Minimum number of expected connections (if less than 5, that connection should be dropped from significance testing!)')
    
    if observed['expected connections'].min() < acceptable_min:
        filtered = observed.loc[observed['expected connections'] < acceptable_min]
        starters_to_remove = filtered['starter cell_type'].unique()
        print('Warning: expected values are below acceptable chi-square threshold:')
        print(filtered['pre-post'])
        print(f'Possible starter types to exclude: \n{starters_to_remove}'),
        print('\nPossible presynaptic types to exclude:')
        print(filtered['non-starter cell_type'].unique())
    else:
        starters_to_remove = []
        print("Values within acceptable threshold! :) ")
    
    return starters_to_remove


def build_enrichment_matrix(
        observed,
        null,
        dataset_id,
        p: float = 0.05
):
    """
    Parameters
    ----------
    observed
    null
    dataset_id
    p
    """
    # Add any missing connections that aren't in the uninfected datasets 
    df_new = pd.DataFrame({
        'pre-post':list((set(observed['pre-post'])).difference(set(null['pre-post']))),
        'proportion of connections': 0})

    # Concatenate both matrices so you have a matrix that's the same size as the observed 
    null = pd.concat([null, df_new])
    
    # Links the proportions from the null matrix to matching combos in the observed matrix
    observed['null proportion of connections'] = observed['pre-post'].map(
        null.set_index('pre-post')['proportion of connections'])

    # Calculate scaled expected connections and categorical distribution test statistics
    observed['expected connections'] = (
        observed['observed connections'].sum() * observed['null proportion of connections'])
    observed['chi-square'] = (
        ((observed['observed connections'] - observed['expected connections']) ** 2) / observed['expected connections'])
    deg_f = (
        observed['non-starter cell_type'].nunique() - 1) * (observed['starter cell_type'].nunique() - 1)

    # find critical value
    x_2 = observed['chi-square'].sum()
    print(
        f'X^2 Test statistic is {x_2}', f'\nDOF is {deg_f}',
        f'\nCritical value for significance on whole table for p<{p}: {chi2.ppf(1 - p, deg_f)}')

    #Calculate standardized pearson residuals
    residuals = observed.pivot(
        index='non-starter cell_type', columns='starter cell_type')['observed connections']
    residuals['non-starter_totals'] = residuals.sum(axis=1)
    residuals.loc['starter_totals'] = residuals.sum(axis=0, numeric_only=True)
    
    #Add residuals back to observed
    observed['non-starter_totals'] = observed['non-starter cell_type'].map(
        residuals['non-starter_totals'])
    observed['starter_totals'] = observed['starter cell_type'].map(
        residuals.T['starter_totals'])

    #Adding residuals to observed
    total_connections = observed['observed connections'].sum()
    observed['standardized_pearsons_residuals'] = (
        observed['observed connections'] - observed['expected connections']) / (np.sqrt(observed['expected connections'] * (1 - (observed['non-starter_totals'] / total_connections)) * (1 - (observed['starter_totals'] / total_connections))))

    #Apply Bonferroni correction to residuals
    count = observed['non-starter cell_type'].nunique() * (
        observed['starter cell_type'].nunique())
    new_alpha = p / count
    critical_score = stats.norm.ppf(1 - new_alpha / 2)
    print(f'Post Hoc testing with Bonferroni correction, number of comparisons: {count}')
    print(f'New alpha level = {new_alpha}')
    print(f'Critical score threshold for multiple comparisons: {critical_score}')
    
    #Adding column to observed that says whether the enrichment is significant
    observed['post_hoc_significance'] = ' '
    mask = abs(observed['standardized_pearsons_residuals']) > critical_score
    observed.loc[mask, 'post_hoc_significance'] = '*'
    
    #Adding pseudocount
    observed['expected connections'] = observed['expected connections'] + 1
    observed['observed connections'] = observed['observed connections'] + 1

    #Calculates the ratio of proportions of observed/null (enrichment score)
    observed['ratio of proportion of connections/null'] = (
        observed['observed connections'] / observed['expected connections'])
    observed['log10 ratio of proportion of connections/null'] = np.log10(
        observed['ratio of proportion of connections/null'])
    
    #Format input for heatmap plot
    pivot = observed.pivot(
        index='non-starter cell_type', columns='starter cell_type')['log10 ratio of proportion of connections/null'].reindex(pivot_types, level=0).T.reindex(cell_types).T
    observed['real observed connections'] = observed['post_hoc_significance'] + (
        observed['observed connections'] - 1).astype(str)
    pivot2 = observed.pivot(
        index='non-starter cell_type', columns='starter cell_type')['real observed connections'].reindex(pivot_types, level=0).T.reindex(pivot_types).T
    
    """
    Is this additional set of +1 offsets duplicated from above?
    """
    # observed['expected connections'] = observed['expected connections'] + 1
    # observed['observed connections'] = observed['observed connections'] + 1
    observed.to_csv(f'./connectivity/matrices/{dataset_id}_observation_table.csv')

    #Scaling is for log10 transformation
    plt.figure(figsize=(15,8))
    sns.heatmap(
        pivot, annot=pivot2, fmt='', square=True, vmin=-2.0, vmax=2, center=0, cmap='bwr')
    plt.ylabel('Non-starter')
    plt.xlabel('Starter')

    #Save figure as an .svg file
    plt.savefig(f'./connectivity/matrices/{dataset_id}_matrix.pdf', bbox_inches = 'tight', format = 'pdf')
    return observed


def build_enrichment_matrix_w_ci(
        observed,
        null,
        dataset_id
):
    """
    Parameters
    ----------
    observed
    null
    dataset_id
    """
    # Add any missing connections that aren't in the uninfected datasets 
    df_new = pd.DataFrame({
        'pre-post': list((set(observed['pre-post'])).difference(set(null['pre-post']))),
        'proportion of connections': 0})

    # Concatenate both matrices so you have a matrix that's the same size as the observed 
    null = pd.concat([null, df_new])
    null_dict = null.set_index(
        'pre-post')[['proportion of connections', 'low95ci', 'high95ci']]

    # Links the proportions from the null matrix to the matching cell type combos in the observed matrix dataframe
    observed['null proportion of connections'] = observed['pre-post'].map(
        null_dict['proportion of connections'])

    # Link confidence intervals
    observed['low95ci'] = observed['pre-post'].map(null_dict['low95ci'])
    observed['high95ci'] = observed['pre-post'].map(null_dict['high95ci'])
    
    # Calculate scaled expected connections
    observed['expected connections'] = (
        observed['observed connections'].sum() * observed['null proportion of connections'])

    # Adding column to observed that says whether the enrichment is significant
    observed['outside_ci95'] = np.where(
        (observed['proportion of connections'] > observed['high95ci']) | (observed['proportion of connections'] < observed['low95ci']), '* ', 'n ')
    
    # Adding pseudocount
    observed['expected connections'] = observed['expected connections'] + 1
    observed['observed connections'] = observed['observed connections'] + 1

    # Calculates the ratio of proportions of observed/null for each cell type (enrichment score)
    observed['ratio of proportion of connections/null'] = (
        (observed['observed connections']) / (observed['expected connections']))
    observed['log10 ratio of proportion of connections/null'] = np.log10(
        observed['ratio of proportion of connections/null'])
    
    #Format input for heatmap plot
    pivot = observed.pivot(
        index='non-starter cell_type', columns='starter cell_type')['log10 ratio of proportion of connections/null'].reindex(pivot_types, level=0).T.reindex(pivot_types).T
    observed['real observed connections'] = (
        observed['outside_ci95'] + (observed['observed connections'] - 1).astype(str))
    pivot2 = observed.pivot(
        index='non-starter cell_type', columns='starter cell_type')['real observed connections'].reindex(pivot_types, level=0).T.reindex(pivot_types).T
    observed.to_csv(f'./connectivity/matrices/{dataset_id}_observation_table.csv')
    
    #Scaling is for log10 transformation
    plt.figure(figsize=(15,8))
    sns.heatmap(pivot, annot=pivot2, fmt='', square=True, vmin=-2.0, vmax=2, center=0, cmap='bwr')
    plt.ylabel('Non-starter')
    plt.xlabel('Starter')

    #Save figure as an .svg file
    plt.savefig(f'./connectivity/matrices/{dataset_id}_matrix.pdf', bbox_inches = 'tight', format = 'pdf')
    return observed


def bootstrap_null_population(
        non_starter_proportions,
        bootstraps,
        cells_to_sample
):
    """
    Parameters
    ----------
    non_starter_proportions
        non-starter proportions generated from uninfected datasets
    bootstraps
        number of times to bootstrap distribution
    cells_to_sample
        number of cells sampled per iteration
    """
    # Expand dataset so you have one entry per cell 
    temp = non_starter_proportions.loc[
        non_starter_proportions.index.repeat(non_starter_proportions['number of cells'])].reset_index(drop=True)

    # Randomly select n barcodes and sum total unique barcodes remaining
    output = []
    for n in range(bootstraps):
        data = pd.DataFrame(
            np.random.choice(temp['non_starter_cell_type'], cells_to_sample, replace=True))
        counts = pd.DataFrame(data[0].value_counts())
        counts.columns = [f'iteration_{n}']
        output.append(counts)
    
    return pd.concat(output, axis=1)


def build_bootstrapped_null_matrix(
        bootstrapped_non_starter_proportions,
        experimental_dataset,
        annotations: str = 'celltype',
        ci: float = 0.95
):
    """
    Parameters
    ----------
    bootstrapped_non_starter_proportions
        non-starter proportions generated from uninfected datasets
    experimental_dataset
        dataset-split (or pooled) w/ starter cell_type and starter CBC columns
    annotations : str, optional
        which column of annotations to use. One of 'broad_class', 'subclass', 'celltype'
    ci : float, optional
        confidence interval to use, default is 95%
    """
    starters = experimental_dataset.drop_duplicates(
        subset='starter CBC').rename(columns={'starter cell_type': 'starter_cell_type'})
    starters['starter_cell_type'] = starters['starter CBC'].map(metadata_df.set_index('cellbarcode')[annotations])

    starter_proportions = starters.groupby(
        'starter_cell_type').count()['datasetid'].reset_index().rename(columns = {'datasetid': 'number of cells'})
    starter_proportions['proportion of celltype in starter cells'] = (
        starter_proportions['number of cells'] / starter_proportions['number of cells'].sum())

    output = []
    for n in bootstrapped_non_starter_proportions.columns.unique():
        # Pull the iteration
        temp = bootstrapped_non_starter_proportions[[n]]
        temp['nonstarter_cell_type'] = temp.index
        
        # Calculate the proportion of nulls
        temp['proportion of celltype in nonstarter cells'] = temp[[n]] / temp[[n]].sum()
        
        # Build null
        temp_null_df = temp.merge(starter_proportions, how='cross')
        temp_null_df['iteration_props'] = (
            temp_null_df['proportion of celltype in nonstarter cells'] * temp_null_df['proportion of celltype in starter cells'])
        
        # Create column of presynaptic to postsynaptic cell type connections
        temp_null_df['pre-post'] = (
            temp_null_df['nonstarter_cell_type'] + "+" + temp_null_df['starter_cell_type'])

        # THIS IS YOUR BOOTSTRAPPED NULL MATRIX!
        # Set index for easy merging
        output.append(temp_null_df.set_index('pre-post')[['iteration_props']])
    
    #Add proportion of connections to dataframe for calculating mean and 95% CI
    output = pd.concat(output, axis=1)
    output['proportion of connections'] = output.mean(axis=1)
    conf = pd.DataFrame(
        output.apply(lambda x: stats.t.interval(ci, len(x)-1, loc=np.mean(x), scale=stats.sem(x)), axis=1))
    conf[['low95ci','high95ci']] = pd.DataFrame(conf[0].tolist(), index=conf.index)
    output = pd.concat([output, conf],axis=1)
    return output


def graph_draw(
        G,
        using_barcodes_df,
        nodescale: int = 20
):
    """
    Parameters
    ----------
    G
    using_barcodes_df
    nodescale : int, optional
    """
    pos = nx.nx_agraph.graphviz_layout(G, prog='neato', root=None, args='')  

    fig, ax = plt.subplots()
    edge_list = list(G.edges())
    nodes_list = list(G.nodes())
    degree_list = list(G.degree())

    # scale node size by max degree among nodes
    if len(edge_list) > 1:
        max_degree = max([x[1] for x in degree_list])
        nodesize = [nodescale * x[1] / max_degree for x in degree_list]

    celltype_dict = dict(zip(using_barcodes_df['CBC'], using_barcodes_df['subclass']))
    colors_dict = {
        'RG': '#8ca252', 'IPC': '#cedb9c', 'EN-Immature': '#8c6d31', 'EN-L2_3-IT': '#e7ba52', 'EN-L4-IT': '#e7cb94', 'EN-Deep Layer': '#843c39', 'Cajal-Retzius cell': '#d6616b', 'IN-CGE': '#e7969c', 'IN-MGE': '#a55194', 'IN-DGE': '#ce6dbd'}

    color_list_nodes = list(
        map(lambda x: colors_dict[x], list(map(lambda x: celltype_dict[x], nodes_list)))) 

    # draw nodes
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=color_list_nodes, node_size=nodesize)

    # color edges by cell type of nonstarter neuron
    if len(edge_list) > 1:
        edges_colors = list(
            map(lambda x: colors_dict[x], list(map(lambda x: celltype_dict[x[0]], edge_list))))      
        nx.draw_networkx_edges(
            G, pos, ax=ax, edgelist=edge_list, width=0.1, node_size=nodesize, edge_color=edges_colors, arrowsize=4, arrows=True)

        # add degree legend
        size_ranges = [np.min(nodesize), np.median(nodesize), np.max(nodesize)]
        size_labels = [
            f'Min: {max_degree / nodescale * size_ranges[0]:.2f}', f'Median: {max_degree / nodescale * size_ranges[1]:.2f}', f'Max: {max_degree / nodescale * size_ranges[2]:.2f}']
        for size, label in zip(size_ranges, size_labels):
            ax.scatter([], [], s=size, color='black', alpha=0.6, label=label)
    
    ax.legend(
        scatterpoints=1, frameon=False, labelspacing=1, loc='upper left', title='Node degree', bbox_to_anchor=(1.00, 1.5))

    # add cell type legend
    for celltype, color in colors_dict.items():
        ax.scatter([], [], s=100, color=color, alpha=0.6, label=celltype)

    # keep title for degree but plot cell type colors
    ax.legend(
        scatterpoints=1, frameon=False, labelspacing=1, loc='upper left',title='Node degree', bbox_to_anchor=(1.00, 1.0))
    return fig


def graph_plot(
        conn_1,
        using_barcodes_df,
        path_to_pdf
):
    """
    Parameters
    ----------
    conn_1
    using_barcodes_df
    path_to_pdf
    """
    G = nx.DiGraph()
    G.add_nodes_from(set(conn_1['non-starter CBC']).union(set(conn_1['starter CBC'])))  
    G.add_edges_from(list(conn_1['connection']))       
            
    fig = graph_draw(G, using_barcodes_df)
    fig.set_size_inches(10,10)
    fig.savefig(path_to_pdf)


def calculate_props(
        network_types,
        dataset_list,
        population
):
    """
    Parameters
    ----------
    dataset_list
        list of categories data are grouped into
    population
        which population to use, pre or post
    """
    final_props = network_types[[population]].drop_duplicates().reset_index()
    for i in dataset_list:
        temp = network_types.loc[network_types['network_size'] == i]
        if population == 'post_subclass':
            temp = temp.drop_duplicates(['starter CBC']).reset_index()
            
        print(len(temp))
        props = pd.DataFrame(temp[population].value_counts()).reset_index()
        props[f'{i}_proportion'] = props['count'] / props['count'].sum()
        final_props = final_props.merge(props[[population, f'{i}_proportion']], on=population, how='left')

    transposed_props = final_props.T
    transposed_props.columns = transposed_props.iloc[1]
    transposed_props = transposed_props[2:].fillna(0)
    return transposed_props

## Import and filter cell type annotation data

In [ ]:
metadata_df = pd.read_csv('./transcriptome/mapped_centroids.csv')

# Remove cells w/ maximum correlation score <= 0.2, update headers, pair down columns
metadata_df = metadata_df.loc[metadata_df['high_score'] > 0.2].rename(
    columns={'type_updated': 'celltype', 'Unnamed: 0':'cellbarcode','dataset_id':'datasetid'})[['cellbarcode','celltype','datasetid']]

#Drop cells mapping to "unknown" cluster
metadata_df = metadata_df[~metadata_df['celltype'].isin(unknown)]

# Adding class and subtype annotations as metadata
metadata_df['broad_class'] = metadata_df['celltype']
metadata_df['subclass'] = metadata_df['celltype']
metadata_df.loc[metadata_df['celltype'].isin(en), 'broad_class'] = 'EN'
metadata_df.loc[metadata_df['celltype'].isin(inh), 'broad_class'] = 'IN'
metadata_df.loc[metadata_df['celltype'].isin(pro), 'broad_class'] = 'IPC'
metadata_df.loc[metadata_df['celltype'].isin(glia), 'broad_class'] = 'Glia'
metadata_df.loc[metadata_df['celltype'].isin(astro), 'subclass'] = 'Astrocyte'
metadata_df.loc[metadata_df['celltype'].isin(opc), 'subclass'] = 'Oligo'
metadata_df.loc[metadata_df['celltype'].isin(rgc), 'subclass'] = 'RG'
metadata_df.loc[metadata_df['celltype'].isin(pro), 'subclass'] = 'IPC'
metadata_df.loc[metadata_df['celltype'].isin(mge), 'subclass'] = 'IN-MGE'
metadata_df.loc[metadata_df['celltype'].isin(cge), 'subclass'] = 'IN-CGE'
metadata_df.loc[metadata_df['celltype'].isin(dge), 'subclass'] = 'IN-DGE'
metadata_df.loc[metadata_df['celltype'].isin(immature), 'subclass'] = 'EN-Immature'
metadata_df.loc[metadata_df['celltype'].isin(deep), 'subclass'] = 'EN-Deep Layer'

#Count number of RVdG+ and uninfected cells
print('Number of uninfected cells/nuclei: ', len(metadata_df[metadata_df.datasetid.isin(['u1','u2'])]))
print('Number of cells/nuclei from RVdG+ datasets: ', len(metadata_df[~metadata_df.datasetid.isin(['u1','u2'])]))

In [ ]:
metadata_df

In [ ]:
#Count non-neuronal populations in RVdG datasets
infected = metadata_df[~metadata_df.datasetid.isin(['u1','u2'])]
print(len(infected))
infected['subclass'].value_counts()

Load in rabies count matrices, assign a matching dataset ID to metadata table from above, and rename to match those cell barcodes. The dataset ID for each sample is also tacked onto the front of the rabies barcode sequence to prevent barcode collision across distinct slices

In [ ]:
barcodes_df = []
for i in dec_list + mar_list:
    data = pd.read_table(f'./barcode_count_matrices/{i}_completecounts.tsv', delimiter='\t')
    data['datasetid'] = i
    data['CBC'] = data['CBC'].str.replace("'", "").str.replace('b', f'{i}_')
    data['barcode'] = f'{i}_' + data['barcode']
    barcodes_df.append(data)

barcodes_df = pd.concat(barcodes_df, ignore_index=True, axis=0)
barcodes_df['barcode_sequence'] = barcodes_df['barcode'].str[3:]  # barcode without dataset id
no_filter_barcodes = barcodes_df.copy()

# Drop any barcodes that don't pass transcriptome QC
barcodes_df = barcodes_df.loc[barcodes_df['CBC'].isin(metadata_df['cellbarcode'])]

# Repeat the same steps as above but for helper counts rather than rabies barcodes
helpers_df = []
for i in dec_list + mar_list:
    data = pd.read_table(f'./barcode_count_matrices/{i}_helperindex.tsv',delimiter = '\t')
    data['datasetid'] = i
    data['CBC'] = data['CBC'].str.replace("'", "").str.replace('b', f'{i}_')
    helpers_df.append(data[['CBC','UMI_Count','datasetid']])

helpers_df = pd.concat(helpers_df, ignore_index = True, axis = 0)

In [ ]:
barcodes_df

In [ ]:
helpers_df

### UpSet plots with all barcodes identified in dataset pools
These plots will let us look at shared barcodes across different experiments, which serves as a proxy for true barcode collision

In [ ]:
for virus, dataset_id_list in {'sadb19': dec_list, 'cvs': mar_list}.items():
    # plot with all barcodes, only barcodes passing UMI threshold
    for name, df in {'no': no_filter_barcodes, 'cell': barcodes_df}.items():
        plot(
            from_contents({k: df.loc[df['datasetid'] == k]['barcode_sequence'].unique().tolist() for k in dataset_id_list}),
            show_counts=True, subset_size="count")
        plt.savefig(
            f'./figs/sfig_conn/{name}_threshold_{virus}_barcode_overlap.pdf', bbox_inches='tight', format='pdf')
        plt.show()

## Barcode thresholding

In [ ]:
# Break into viral library matched experiments for SBARRO thresholding
dec_exp = barcodes_df.loc[barcodes_df['datasetid'].isin(dec_list)]
mar_exp = barcodes_df.loc[barcodes_df['datasetid'].isin(mar_list)]

# For Shin & Urbanek et al., we used UMI thresholding of 2
dec_exp = umi_threshold(dec_exp, umi=2)
mar_exp = umi_threshold(mar_exp, umi=2)

### Format viral diversity libraries for SBARRO pipeline

Viral diversity libraries are generated from sequencing the rabies genome. Because they are the reverse complement of the sequences derived from polyA capture with Pip-seq, we need to make a bit key that allows us to switch between strands of barcode sequences. The resulting dataframe is derived from our bit sequences and should have four columns:
1. original = barcode number, forward orientation
2. reverse = barcode number, reverse orientation
3. original_seq = barcode sequence, forward orientation
4. reverse_seq = barcide sequence, reverse orientation

In [ ]:
remapping_bits = pd.read_csv('./remapping_bits.csv', header=None)
remapping_bits.columns = ['original', 'reverse']
remapping_bits['original'] = remapping_bits['original'].str.replace(">","")
remapping_bits['reverse'] = remapping_bits['reverse'].str.replace(">","")

fragments = remapping_bits.iloc[::2]
bits = remapping_bits.iloc[1::2]
bit_key = pd.concat([fragments, bits.set_index(fragments.index)], axis = 1)
bit_key.columns = ['original','reverse','original_seq','reverse_seq']

# Set original forward indexing as key
original_key = bit_key.set_index('original')

In [ ]:
original_key

In [ ]:
# Read-in and format viral diversity library for SAD B19 experiments
dec2023_sadb = pd.read_csv(
    './used_in_experiments/sad_rep.csv')[['barcode','UMI_Count']].rename(
    columns={'barcode': 'viral_barcode', 'UMI_Count': 'viral_total_UMIs'})

# Add barcode proportions and rank barcodes by prevalence
dec2023_sadb['%UMI'] = (
    dec2023_sadb['viral_total_UMIs'] * 100 / dec2023_sadb['viral_total_UMIs'].sum())
dec2023_sadb = dec2023_sadb.sort_values(by='%UMI', ascending=False).reset_index()
dec2023_sadb['rank'] = dec2023_sadb.index

dec2023_sadb[['bit1', 'bit2', 'bit3']] = dec2023_sadb['viral_barcode'].str.split(
    '-', expand=True)

# # realigned these using forward orientation, don't need to be reformatted
# dec2023_sadb['original_barcode'] = (
#     dec2023_sadb['bit3'].map(remapping_bits.set_index('reverse')['original']) + "-" +  # original bit1
#     dec2023_sadb['bit2'].map(remapping_bits.set_index('reverse')['original']) + "-" +  # original bit2
#     dec2023_sadb['bit1'].map(remapping_bits.set_index('reverse')['original']))  # original bit3
dec2023_sadb['original_barcode'] = dec2023_sadb['viral_barcode']

# Explode dataframe so each viral barcode has as many entries as UMIs in the complexity library
exp_dec2023_sadb = dec2023_sadb.loc[
    dec2023_sadb.index.repeat(dec2023_sadb['viral_total_UMIs']), ['original_barcode']].reset_index(drop=False)

In [ ]:
exp_dec2023_sadb

In [ ]:
# Repeating the steps above for the CVS-N2c complexity library
# For March 2024, CVS libraries
mar2024_cvs = pd.read_csv('./used_in_experiments/cvs_rep.csv')[['barcode','UMI_Count']].rename(
    columns={'barcode': 'viral_barcode', 'UMI_Count': 'viral_total_UMIs'})

# Add metrics barcode proportions and rank barcodes by prevalence
mar2024_cvs['%UMI'] = (
    mar2024_cvs['viral_total_UMIs'] / mar2024_cvs['viral_total_UMIs'].sum()) * 100
mar2024_cvs = mar2024_cvs.sort_values(by='%UMI', ascending=False).reset_index()
mar2024_cvs['rank'] = mar2024_cvs.index

# Formatting
mar2024_cvs[['bit1', 'bit2', 'bit3']] = mar2024_cvs['viral_barcode'].str.split('-', expand=True)
mar2024_cvs['original_barcode'] = (
    mar2024_cvs['bit3'].map(remapping_bits.set_index('reverse')['original']) + "-" +  # original bit1
    mar2024_cvs['bit2'].map(remapping_bits.set_index('reverse')['original']) + "-" +  # original bit2
    mar2024_cvs['bit1'].map(remapping_bits.set_index('reverse')['original']))  # original bit3

# Expand library
exp_mar2024_cvs = mar2024_cvs.loc[
    mar2024_cvs.index.repeat(mar2024_cvs['viral_total_UMIs']), ['original_barcode']].reset_index(drop=False)

In [ ]:
exp_mar2024_cvs

### Model single starter infection with SBARRO
Steps for SBARRO:
1) For each experiment create a distribution of VBC drawing simulations (n=10, with replacement) required to match VBCs observed in at least 2 cells in the experiment
2) The median value from step 1 is set as the founder infection estimates for each experiment
3) For each barcode, calculate its FI score:
    - FI = log10(p)/log10(1-f)
    - p = probability of avoiding a second founder infection derived from simulations
    - f = frequency in viral diversity library
4) Retain barcode sets where FI trust score > FI estimates (step 2)

The “FI trust” score equates to the number of spreading founder infections that could in theory occur for that VBC or VBC set before a second founder infection was expected (at a given probability) by leveraging library VBC frequencies

In [ ]:
# Calculate FI trust score threshold for each dataset bootstrapping 10 times
np.random.seed(13)
dec_exp = {i: dec_exp.loc[dec_exp['datasetid'] == i] for i in dec_list}
mar_exp = {i: mar_exp.loc[mar_exp['datasetid'] == i] for i in mar_list}
fi_trust_dec = {i: calculate_fi_trust(dec_exp[i], exp_dec2023_sadb) for i in dec_list}
fi_trust_mar = {i: calculate_fi_trust(mar_exp[i], exp_mar2024_cvs) for i in mar_list}
del exp_dec2023_sadb, exp_mar2024_cvs

In [ ]:
# Calculate FI Trust score for each viral barcode with different acceptable dual infection risk (p)
dec2023_sadb['vbc_proportion'] = (
    dec2023_sadb['viral_total_UMIs'] / dec2023_sadb['viral_total_UMIs'].sum())
mar2024_cvs['vbc_proportion'] = (
    mar2024_cvs['viral_total_UMIs'] / mar2024_cvs['viral_total_UMIs'].sum())

for p in (0.9, 0.925, 0.95, 0.975, 0.99):
    dec2023_sadb[f'p = {p}'] = np.log10(p) / (np.log10(1 - dec2023_sadb['vbc_proportion']))
    mar2024_cvs[f'p = {p}'] = np.log10(p) / (np.log10(1 - mar2024_cvs['vbc_proportion']))

# Plot individual barcodes versus whole sample library threshold
colors = {
    'p = 0.9': 'royalblue', 'p = 0.925': 'mediumseagreen', 'p = 0.95': 'darkkhaki', 'p = 0.975': 'sandybrown', 'p = 0.99': 'firebrick'}

# December 2023, SADB19
for p, c in colors.items():
    plt.plot(dec2023_sadb['rank'], dec2023_sadb[p], label=p, color=c, linewidth=1.5)

plt.xlabel("Log10(Barcode Rank)")
plt.ylabel("FI Trust Score")
plt.title("December 2023, SADB19")
plt.axhline(y=fi_trust_dec['s1'], color='black', linestyle='solid', label="S1 FI Threshold")
plt.axhline(y=fi_trust_dec['s2'], color='black', linestyle='dotted', label="S2 FI Threshold")
plt.axhline(y=fi_trust_dec['s3'], color='black', linestyle='dashed', label="S3 FI Threshold")
plt.axhline(y=fi_trust_dec['s4'], color='black', linestyle='dashdot', label="S4 FI Threshold")
plt.axhline(y=fi_trust_dec['s5'], color='black', linestyle=(0, (3, 5, 1, 5, 1, 5)), label="S5 FI Threshold")
plt.yscale('log')
plt.xscale('log')
plt.legend(loc='upper left')
plt.draw()
plt.show()

# March 2024, CVS
for p, c in colors.items():
    plt.plot(mar2024_cvs['rank'], mar2024_cvs[p], label=p, color=c, linewidth=1.5)

plt.xlabel("Log10(Barcode Rank)")
plt.ylabel("FI Trust Score")
plt.title("March 2024, CVS-N2c")
plt.axhline(y=fi_trust_mar['c1'], color='black',linestyle='solid',label="C1 FI Threshold")
plt.axhline(y=fi_trust_mar['c2'], color='black',linestyle='dotted',label="C2 FI Threshold")
plt.axhline(y=fi_trust_mar['c3'], color='black',linestyle='dashed',label="C3 FI Threshold")
plt.axhline(y=fi_trust_mar['c4'], color='black',linestyle='dashdot',label="C4 FI Threshold")
plt.axhline(y=fi_trust_mar['n1'], color='grey',linestyle='solid',label="N1 FI Threshold")
plt.axhline(y=fi_trust_mar['n2'], color='grey',linestyle='dotted',label="N2 FI Threshold")
plt.axhline(y=fi_trust_mar['n3'], color='grey',linestyle='dashed',label="N3 FI Threshold")
plt.axhline(y=fi_trust_mar['n4'], color='grey',linestyle='dashdot',label="N4 FI Threshold")
plt.yscale('log')
plt.xscale('log')
plt.legend(loc='upper left')
plt.draw()
plt.show()

In [ ]:
# December 2023 datasets
dec_exp = {
    i: threshold_fi_trust(df, fi_trust_dec[i], dec2023_sadb) for i, df in dec_exp.items()}

# March 2024 datasets
mar_exp = {
    i: threshold_fi_trust(df, fi_trust_mar[i], mar2024_cvs) for i, df in mar_exp.items()}

# Plot 
plt.figure(figsize=(8, 1))
plt.scatter(
    np.log10(dec2023_sadb['rank']), dec2023_sadb['%UMI'], color='lightgrey', linewidth=1.5, alpha=1)
for i, c in zip(dec_list, ['green', 'blue', 'indigo', 'orange', 'red']):
    overlap = dec2023_sadb[
        dec2023_sadb['original_barcode'].isin(dec_exp[i]['barcode_sequence'])]
    plt.scatter(
        np.log10(overlap['rank']), overlap['%UMI'], label=i.upper(), color=c, linewidth=1.5, marker=',', s=2)

plt.xlabel("Log10(Barcode Rank)")
plt.ylabel("%UMI")
plt.draw()
plt.show()

plt.figure(figsize=(8, 1))
plt.scatter(
    np.log10(mar2024_cvs['rank']), mar2024_cvs['%UMI'], color='lightgrey', linewidth=1.5, alpha=1)
for i, c in zip([i for i in mar_list if 'c' in i], ['green', 'blue', 'indigo', 'red']):
    overlap = mar2024_cvs[
        mar2024_cvs['original_barcode'].isin(mar_exp[i]['barcode_sequence'])]
    plt.scatter(
        np.log10(overlap['rank']), overlap['%UMI'], label=i.upper(), color=c, linewidth=1.5, marker=',', s=2)
    plt.scatter(np.log10(overlap['rank']), overlap['%UMI'], label = i.upper(), color=c, linewidth=1.5, marker=',', s=2)

plt.xlabel("Log10(Barcode Rank)")
plt.ylabel("%UMI")
plt.draw()
plt.show()

plt.figure(figsize=(8, 1))
plt.scatter(
    np.log10(mar2024_cvs['rank']), mar2024_cvs['%UMI'], color='lightgrey', linewidth=1.5, alpha=1)
for i, c in zip([i for i in mar_list if 'n' in i], ['green', 'blue', 'indigo', 'red']):
    overlap = mar2024_cvs[
        mar2024_cvs['original_barcode'].isin(mar_exp[i]['barcode_sequence'])]
    plt.scatter(
        np.log10(overlap['rank']), overlap['%UMI'], label=i.upper(), color=c, linewidth=1.5, marker=',', s=2)
    plt.scatter(np.log10(overlap['rank']), overlap['%UMI'], label = i.upper(), color=c, linewidth=1.5, marker=',', s=2)

plt.xlabel("Log10(Barcode Rank)")
plt.ylabel("%UMI")
plt.draw()
plt.show()

# Post thresholding, concatenate all thresholded dataframes together
thresholded_barcodes_df = pd.concat(
    [dec_exp[i] for i in dec_list] + [mar_exp[i] for i in mar_list], ignore_index=True, axis=0)
del dec_exp, mar_exp, dec2023_sadb, mar2024_cvs

In [ ]:
thresholded_barcodes_df

### Upset Plot for thresholded barcodes

In [ ]:
# Print some stats
print(
    'Number of unique barcodes pre-thresholding:', barcodes_df['barcode'].nunique(),
    '\nNumber of unique barcodes post-thresholding:', thresholded_barcodes_df['barcode'].nunique(),
    '\nNumber of cells w/ barcodes pre-thresholding:', barcodes_df['CBC'].nunique(),
    '\nNumber of cells w/ barcodes post-thresholding:', thresholded_barcodes_df['CBC'].nunique())

# Pull barcodes passing UMI threshold in real cells
upset_input = thresholded_barcodes_df.loc[
    thresholded_barcodes_df['CBC'].isin(metadata_df['cellbarcode'])]

# Generating input to upset plot by pulling each dataset
barcode_overlap = from_contents({
    k: upset_input.loc[upset_input['datasetid'] == k, 'barcode_sequence'].unique()
    for k in dec_list})
plot(barcode_overlap, show_counts=True, subset_size="count")
matplotlib.rc('font', family='Arial')
plt.show()  

barcode_overlap = from_contents({
    k: upset_input.loc[upset_input['datasetid'] == k, 'barcode_sequence'].unique()
    for k in mar_list})
plot(barcode_overlap, show_counts=True, subset_size="count")
matplotlib.rc('font', family='Arial')
plt.show()
del barcode_overlap, upset_input

## Connectivity Analysis
### Formatting dataframe for input into analysis

Inputs include:

    1) metadata_df that has matching cell barcodes, type annotations, and datasetIDs at a minimum
    2) thresholded_barcodes_df generated above, which should include cell barcodes, barcode sequences, and barcode UMIs
    3) thresholded_helpers_df generated above, which should include cell barcodes and number of helper UMIs
    

In [ ]:
# Run process_barcodes_df() function on pooled datasets
# Threshold helper_df on UMIs--for Shin & Urbanek et al., we used a UMI threshold of 1
processed_barcodes_df = process_barcodes_df(
    metadata_df, 'celltype', thresholded_barcodes_df, helpers_df.loc[helpers_df['UMI_Count'] >= 1])

In [ ]:
processed_barcodes_df

In [ ]:
# Determine number of barcodes found in single starter cell and >= one non-starter cell
keep_list = processed_barcodes_df.loc[
    (processed_barcodes_df['single_starter_barcode'] == 'y') & (processed_barcodes_df['starter/nonstarter barcode'] == 'both')]
print(
    'Number of unique rabies barcodes found in single starter cells + non-starter cells: ', keep_list['barcode'].nunique())
print(
    'Number of starter cells: ', processed_barcodes_df[processed_barcodes_df.helper == 'starter']['CBC'].nunique())
print(
    'Number of non-starter cells: ', processed_barcodes_df[processed_barcodes_df.helper == 'nonstarter']['CBC'].nunique())

# Identify total number of presynaptic cells per barcode in single starter cells for potential thresholding later
keep_list = keep_list.loc[keep_list['helper'] != 'starter'].groupby(
    'barcode')['CBC'].nunique()
keep_list = keep_list.loc[keep_list < 50].index

In [ ]:
# Any other features to add? Use a dictionary like this:
processed_barcodes_df['broad_class'] = processed_barcodes_df['CBC'].map(
    metadata_df.set_index('cellbarcode')['broad_class'])
processed_barcodes_df['subclass'] = processed_barcodes_df['CBC'].map(
    metadata_df.set_index('cellbarcode')['subclass'])

In [ ]:
processed_barcodes_df

In [ ]:
#Export to look at cell type proportions
processed_barcodes_df[['CBC', 'subclass']].drop_duplicates(subset=['CBC'])['subclass'].value_counts()

### Calculate barcode metrics for pooled datasets

In [ ]:
# Count up number of helpers per barcode
starter_count = processed_barcodes_df.groupby(
    'barcode').value_counts(['helper']).reset_index()

# Drop rows corresponding to nonstarter populations
starter_count = starter_count[starter_count['helper'] != 'nonstarter']
n_starter = starter_count['barcode'].nunique()
print(f'Number of rabies barcodes w/ at least 1 starter: {n_starter}')

# Add non-starter barcodes
zeros = pd.DataFrame(
    0, index=range(processed_barcodes_df['barcode'].nunique() - n_starter), columns=['barcode','helper','count'])

print(f'Number of rabies barcodes w/ no starters: {len(zeros)}')

# Collapse 10+ starters into the 10 bin
starter_count = pd.concat([starter_count, zeros])
starter_count.loc[starter_count['count'] > 9, 'count'] = 10

print('Distribution of number of starters identified per unique barcode:')
starter_count['count'].value_counts()

In [ ]:
starter_count

In [ ]:
# Plot histogram of number of starters per barcode network:
plt.figure().set_figheight(3)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)
fig.subplots_adjust(hspace=0.1)  # adjust space between Axes

# plot the same data on both Axes
ax1.hist((starter_count['count']),bins=np.arange(0, 12, 1),color='forestgreen')
ax2.hist((starter_count['count']),bins=np.arange(0, 12, 1),color='forestgreen')

# zoom-in / limit the view to different portions of the data
ax1.set_ylim(2000, 15000)  # outliers only
ax2.set_ylim(0, 1500)  # most of the data
ax1.set_xlim(0, 11)  # outliers only
ax2.set_xlim(0, 11)  # most of the data
ax1.spines.bottom.set_visible(False)
ax2.spines.top.set_visible(False)
ax1.xaxis.tick_top()
ax1.tick_params(labeltop=False)  # don't put tick labels at the top
ax2.xaxis.tick_bottom()
plt.show()

#Plot histogram of number of starters per barcode network w/o gap in y-scale:
plt.figure().set_figheight(3)
plt.hist(starter_count['count'],bins=np.arange(0, 12, 1),color='forestgreen')
plt.show()

In [ ]:
# Determine number of unique barcodes overall
print(
    'Number of unique rabies barcodes detected in dataset: ', processed_barcodes_df['barcode'].nunique())
print(
    'Number of unique rabies barcodes detected across starter cells: ', processed_barcodes_df.loc[(processed_barcodes_df['helper'] == 'starter')]['barcode'].nunique())
print(
    'Number of unique rabies barcodes not found in single starter cells: ', processed_barcodes_df.loc[(processed_barcodes_df['single_starter_barcode'] == 'n') & (processed_barcodes_df['helper'] == 'starter')]['barcode'].nunique())
print(
    'Number of unique rabies barcodes found in single starter cells: ', processed_barcodes_df.loc[(processed_barcodes_df['single_starter_barcode'] == 'y')]['barcode'].nunique())
print(
    'Number of unique rabies barcodes found in single starter cells + non-starter cells: ', processed_barcodes_df.loc[(processed_barcodes_df['single_starter_barcode'] == 'y') & (processed_barcodes_df['starter/nonstarter barcode'] == 'both')]['barcode'].nunique())

# Determine distribution of number of cells per barcode after removing all barcodes with either no or more than 1 starter
processed_barcodes_df.loc[processed_barcodes_df['single_starter_barcode'] == 'y'].groupby(
    'barcode').count()['helper'].reset_index().rename(columns = {'helper':'number of cells with barcode'}).sort_values('number of cells with barcode',ascending = False)

### Identify single starter, directed networks

##### Filter out networks that are likely to have come from starter drop-out (barcode is in >n presynaptic cells)

This step is optional! For Shin & Urbanek et al., we set n to 50

In [ ]:
# Trim processed_barcodes_df down to keep_list derived in steps above
processed_barcodes_df = processed_barcodes_df.loc[
    processed_barcodes_df['barcode'].isin(keep_list)]

# Pull all rows that have a rabies barcode in one starter cell and appear in both starter and non-starter cells
df = processed_barcodes_df.loc[(processed_barcodes_df['single_starter_barcode'] == 'y') & (processed_barcodes_df['starter/nonstarter barcode'] == 'both')]

# Create dictionary to link cell barcodes from processed_barcodes_df to their correspondent cell types
cell_type_dict = df[['CBC', 'celltype']].drop_duplicates().set_index('CBC')['celltype']

# Pull starter/non-starter cell populations and make sets of all their rabies barcodes
starter_df = df.loc[df['helper'] == 'starter'].groupby(
    ['datasetid','CBC'])['barcode'].apply(set).reset_index()
starter_df['matching_cells'] = [[] for _ in range(len(starter_df.index))]

# Link each starter to all non-starters that share at least one rabies barcode
cbc_overlap = df.loc[df['helper'] == 'starter', ['CBC', 'barcode']].merge(
    df.loc[df['helper'] == 'nonstarter', ['CBC', 'barcode']], on='barcode', suffixes=('_starter', '_nonstarter')).groupby('CBC_starter')['CBC_nonstarter'].unique().apply(list)
starter_df = starter_df.set_index('CBC')
starter_df.loc[cbc_overlap.index, 'matching_cells'] = cbc_overlap

# Annotate columns to match starter vs. non-starter assignment
starter_df = starter_df.reset_index(
    drop=False).rename(columns={'CBC':'starter CBC', 'barcode':'starter barcodes', 'matching_cells':'non-starter CBC'})

# Use cell type dictionary to assign starter cell types to each starter cell barcode
starter_df['starter cell_type'] = starter_df['starter CBC'].map(cell_type_dict)

# Build input dataframe for connectivity matrix
conn_0_compiling = starter_df[
    ['datasetid', 'starter cell_type', 'starter CBC', 'non-starter CBC']]

# Calculate the number of non-starters sharing a barcode with each starter cell and save to the dataframe
conn_0_compiling['number of non-starters associated'] = conn_0_compiling[
    'non-starter CBC'].apply(lambda x: len(x))

# Calculate the number of total cells connected to a single starter cell (it should be the 'number of non-starters associated' column +1)
conn_0_compiling['number of cells in single-starter network'] = conn_0_compiling[
    'number of non-starters associated'] + 1

# Explode non-starter cell barcode sets so now each non-starter cell gets its own line in the dataframe
conn_1_compiling = starter_df.explode('non-starter CBC')[
    ['datasetid', 'starter cell_type', 'starter CBC', 'non-starter CBC']]

# Assign non-starter cell types based on matching dictionary entry
conn_1_compiling['non-starter cell_type'] = conn_1_compiling[
    'non-starter CBC'].map(cell_type_dict)

# Establish cell type connectivity by merging non-starter (presynaptic) to starter (postsynaptic) cell types
conn_1_compiling['conn_type'] = (
    conn_1_compiling['non-starter cell_type'] + '+' + conn_1_compiling['starter cell_type'])

# Annotate the exact cell to cell connection captured
conn_1_compiling['connection'] = conn_1_compiling.apply(
    lambda x: (x['non-starter CBC'], x['starter CBC'], {'w': 1}), axis=1)

In [ ]:
starter_df

In [ ]:
# conn_0_compiling tells you all of the collapsed single starter networks
print(f'Total number of collapsed networks: {len(conn_0_compiling)}')

# conn_1_compiling tells you the total number of cell:cell connections from single starter networks:
print(f'Total single starter connections: {len(conn_1_compiling)}')

### Thresholding on duplicate connections

This step is optional--if there are a lot of barcodes per cell in your data (and a lot of spread), you may be able to capture sets of barcodes shared between cells rather than individual barcodes. We do not see this in the data in Shin & Urbanek et al., but we did generate some supplementary plots to show what the connectivity matrix looks like with this thresholding applied

In [ ]:
# Check distribution of number of barcodes per connection 
single_starter_network = processed_barcodes_df.loc[
    (processed_barcodes_df['single_starter_barcode'] == 'y') & (processed_barcodes_df['starter/nonstarter barcode'] == 'both')]

bc_per_connection = []
for i in single_starter_network['barcode'].unique():
    temp = single_starter_network.loc[single_starter_network['barcode'] == i]
    starter = temp.loc[temp['helper'] == 'starter', ['CBC']].rename(
        columns={'CBC': 'starter'})
    nonstarter = temp.loc[temp['helper'] == 'nonstarter', ['CBC']].rename(
        columns={'CBC': 'nonstarter'})
    for n in starter['starter'].unique():
        nonstarter['starter'] = n
    
    nonstarter['connection'] = nonstarter['nonstarter'] + "+" + nonstarter['starter']
    bc_per_connection.append(nonstarter)

bc_per_connection = pd.concat(bc_per_connection)

# Get distribution of barcodes per connection
bc_per_connection['connection'].nunique()
bc_counts = pd.DataFrame(bc_per_connection['connection'].value_counts())

In [ ]:
single_starter_network

In [ ]:
bc_counts

In [ ]:
# Plot distribution of number of barcodes per connection
plt.figure().set_figheight(5)
plt.hist(
    bc_counts['count'], bins=np.arange(0, bc_counts['count'].max() + 1.5, 1), alpha=0.7, weights=np.ones(len(bc_counts['count'])) / len(bc_counts['count']))
plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
plt.xlabel('Barcodes per connection')
# plt.ylabel('Frequency')
# plt.xlim([-1, 100])
# plt.ylim([0, 100])
# plt.yscale('log')
# plt.legend()
# plt.title('UMIs per Barcode:CBC Combo')
# plt.savefig('../figs/sfig_conn/barcodes_per_connection_1.svg', bbox_inches = 'tight', format = 'svg')
plt.show()

DO NOT RUN THE FOLLOWING CELL UNLESS YOU WANT TO INCLUDE THIS THRESHOLDING IN YOUR ANALYSIS AS IT WILL CHANGE YOUR CONN_1_COMPILING!

In [ ]:
# Pull conn_1_compiling rows w/ multiple barcodes
conn_1_compiling['combo'] = (
    conn_1_compiling['non-starter CBC'] + '+' + conn_1_compiling['starter CBC'])
conn_1_compiling = conn_1_compiling.loc[
    conn_1_compiling['combo'].isin(bc_counts.loc[bc_counts['count'] > 1].index)]
conn_1_compiling

### Plot distribution of network size

In [ ]:
# Plot distribution of number of cells/nuclei per single-starter network
plt.figure().set_figheight(5)

# Calculate number of starters per barcode
network_count = conn_1_compiling.groupby(
    'starter CBC')['non-starter CBC'].nunique().reset_index()
network_count.loc[network_count['non-starter CBC'] > 50, 'non-starter CBC'] = 50

# Print some stats about single starter network distribution
print(f'Median of distribution: ', network_count['non-starter CBC'].median())
print(f'Total number of networks: {len(network_count)}')

plt.hist(
    network_count['non-starter CBC'], bins=np.arange(0, network_count['non-starter CBC'].max() + .05, 1), color='#df1864', alpha=0.6)

plt.xlabel('Number of cells/nuclei')
plt.ylabel('Single-starter Networks')
plt.xlim([-.1, 50])
# plt.yscale('log')
# plt.title('UMIs per Barcode:CBC Combo')
# plt.savefig("../figs/sfig_conn/102025_network_size.pdf",bbox_inches='tight', format='pdf')
plt.show()

##### If filtering connections with multiple barcodes (barcode sets rather than single barcode spread), need to run the following two steps to reformat conn_1_compiling. Otherwise, you can ignore it!

In [ ]:
#If filtering for connections with multiple barcodes...
conn_1_compiling['cell_cell'] = (
    conn_1_compiling['non-starter CBC'] + "+" + conn_1_compiling['starter CBC'])
bc_counts = bc_counts[bc_counts['count'] >= 2]
conn_1_compiling = conn_1_compiling.loc[conn_1_compiling['cell_cell'].isin(bc_counts.index)]

In [ ]:
bc_counts

In [ ]:
conn_1_compiling

### Finish formatting for visualization

In [ ]:
# Any other features to add? Use a dictionary like this:
feature_dict = metadata_df.set_index('cellbarcode')[['broad_class', 'subclass', 'celltype']]
for c in ('broad_class', 'subclass', 'celltype'):
    conn_1_compiling[f'pre_{c}'] = conn_1_compiling['non-starter CBC'].map(feature_dict[c])
    conn_1_compiling[f'post_{c}'] = conn_1_compiling['starter CBC'].map(feature_dict[c])

# Making subclass to subclass connectivity column and exporting final networks
conn_1_compiling['subclass_conn'] = (
    conn_1_compiling['pre_subclass'] + '+' + conn_1_compiling['post_subclass'])
# conn_1_compiling.to_csv('./connectivity/sbarro_990_single_starter_networks.csv')

In [ ]:
conn_1_compiling

In [ ]:
# For visualization, removing rows with low-proportion annotations (LAMP5, vascular, microglia)
drop = ['Vascular', 'IN-Mix-LAMP5', 'Microglia']
subset_conn_1_compiling = conn_1_compiling[
    ~(conn_1_compiling['pre_celltype'].isin(drop) | conn_1_compiling['post_celltype'].isin(drop))]

# For doing drop-out simulations later
drop = ['Oligo', 'Astrocyte']
non_neuronal = subset_conn_1_compiling[
    ~(subset_conn_1_compiling['pre_subclass'].isin(drop) | subset_conn_1_compiling['post_subclass'].isin(drop))]
combo_list = non_neuronal['subclass_conn'].unique()

# Exporting for Sankey plotting w/ SankeyMatic
subset_conn_1_compiling['subclass_conn'].value_counts(
    ).to_frame('count').to_csv('./connectivity/sankey_input.csv')

In [ ]:
# Sort networks from largest to smallest network sizes
conn_0_compiling = conn_0_compiling.sort_values(
    'number of cells in single-starter network', ascending = False)

# Give each network its own ID
conn_0_compiling['starter network id'] = np.arange(len(conn_0_compiling)) + 1

# Build 'CBC' column that includes all cells within the network
conn_0_compiling['starter CBC'] = conn_0_compiling['starter CBC'].apply(lambda x: [x])
conn_0_compiling['CBC'] = conn_0_compiling.apply(
    lambda x: x['starter CBC'] + x['non-starter CBC'], axis=1)

# Pull all single starter networks that also have non-starter partners
df = processed_barcodes_df.loc[
    (processed_barcodes_df['single_starter_barcode'] == 'y') & (processed_barcodes_df['duplicate?'] == True)][['CBC', 'celltype', 'helper']].drop_duplicates().set_index('CBC')

# Build dataframe that has a separate entry for every cell (both starter and non-starter) in single starter networks
df2 = conn_0_compiling.explode('CBC')[
    ['datasetid','starter network id','number of cells in single-starter network','CBC']]

# Assign cell types to every cell in single starter networks
df2['celltype'] = df2['CBC'].map(df['celltype'])

# Assign starter status based on the presence of helper infection for every cell in single starter networks
df2['starter status'] = df2['CBC'].map(df['helper'])

# Swap nonstarter to non-starter and rerun starter status assignment
df2['starter status'] = df2['starter status'].map(
    {'nonstarter':'non-starter','starter':'starter'})

# Adding features back to table
feature_dict = metadata_df.set_index('cellbarcode')[['broad_class', 'subclass']]
for c in ('broad_class', 'subclass'):
    df2[c] = df2['CBC'].map(feature_dict[c])

In [ ]:
df2

In [ ]:
# Export processed_barcodes_df and metadata_df for starter drop-out simulations
processed_barcodes_df.to_csv('./connectivity/2026_processed_barcodes_df.csv')
metadata_df.to_csv('./connectivity/2026_metadata_df.csv')

### Enrichment matrices

In [ ]:
en_subclasses = [
    'EN-L2_3-IT', 'EN-Immature','EN-Deep Layer','EN-L4-IT','Cajal-Retzius cell']
print('Total single starter barcode networks before removing glial populations:')
print(conn_1_compiling['starter CBC'].nunique())

print('Total number of presynaptic cells:')
print(conn_1_compiling['non-starter CBC'].nunique())

print('Number of presynaptic excitatory neurons:')
print(
    conn_1_compiling[conn_1_compiling['pre_subclass'].isin(en_subclasses)]['non-starter CBC'].nunique())

print('Total number of postsynaptic cells:')
print(conn_1_compiling['starter CBC'].nunique())

print('Number of postsynaptic excitatory neurons:')
print(
    conn_1_compiling[conn_1_compiling['post_subclass'].isin(en_subclasses)]['starter CBC'].nunique())

# Dropping low-represented in centroid mapping neuronal types
subset_metadata_df = metadata_df[metadata_df['subclass'] != 'IN-Mix-LAMP5']
subset_conn_1_compiling = conn_1_compiling[
    (conn_1_compiling['pre_subclass'] != 'IN-Mix-LAMP5') & (conn_1_compiling['post_subclass'] != 'IN-Mix-LAMP5')]

# Calculate non-neuronal connectivity
types_to_pull=['Oligo', 'Astrocyte', 'RG', 'Vascular', 'IPC']
total = 0
for t in types_to_pull:
    t_count = (
            (subset_conn_1_compiling['pre_subclass'] == t) | (subset_conn_1_compiling['pre_subclass'] == t)).sum()
    total += t_count
    print(f'Non-neuronal types include: {types_to_pull}')
    print(f'Number of non-neuronal connections: {t_count}/{len(subset_conn_1_compiling)}')
    print(
        f'Percentage of total connections that are non-neuronal: {t_count / len(subset_conn_1_compiling) * 100}')
    
print(
    'Percentage of total connections that are non-neuronal: {total / len(subset_conn_1_compiling) * 100}')

# Dropping non-neuronal and low represented populations from dataframes
drop = ['Microglia', 'Vascular', 'Astrocyte', 'Oligo', 'IN-Mix-LAMP5']
subset_metadata_df = metadata_df[
    ~(metadata_df['subclass'].isin(drop) | metadata_df['celltype'].isin(drop))]
subset_conn_1_compiling = conn_1_compiling[
    ~(conn_1_compiling['pre_subclass'].isin(drop) | conn_1_compiling['post_subclass'].isin(drop))]

In [ ]:
print('Total starter cells after all thresholding and subsetting')
print(subset_conn_1_compiling['starter CBC'].nunique())

print('Total presynaptic cells after all thresholding and subsetting')
print(subset_conn_1_compiling['non-starter CBC'].nunique())

print(f'Total number of connections: {len(subset_conn_1_compiling)}')
subset_conn_1_compiling

#### Calculating significance w/ Chi-Square
This code will plot all populations, but only calculate significance on rows and columns with sufficient counts

##### Pooled matrix

In [ ]:
# Build null
uninfected_df = subset_metadata_df.loc[
    subset_metadata_df['datasetid'].isin(['u1', 'u2'])].groupby('subclass').count()['datasetid'].reset_index()
uninfected_df['proportion of celltype in nonstarter cells'] = uninfected_df[
    'datasetid'] / uninfected_df['datasetid'].sum()

# Rename 'datasetid' metadata column since it was overwritten by .count() function in preceding lines
uninfected_df = uninfected_df.rename(
    columns={'datasetid': 'number of cells', 'subclass':'non_starter_cell_type'})

# Clarify cell type assignments here should be used to generate non_starter null matrix populations
pooled_null = build_null_matrix(
    uninfected_df, subset_conn_1_compiling, metadata_df, annotations='subclass')

# Build observed
pooled_obs = build_obs_matrix(
    subset_conn_1_compiling, pooled_null, metadata_df, annotations='subclass')

# Check expected numbers for chi-square testing
starters_excluded = check_minimum_expected_connections(pooled_obs, pooled_null, 5)

# If there are no starters to exclude (expected counts are ≥5 for all connections, specify empty array for starters_excluded:
# starters_excluded=[]

# Generate enrichment matrix
pooled_cell_sub_enrichment = build_enrichment_matrix(
    pooled_obs, pooled_null, 'drop_10', starters_excluded)

##### Split by age